# CSE 144 Final Project | Perception Encoder (PE)
**Due: June 10, 2026**

This notebook trains Meta's Perception Encoder (ViT-Large, 336x336) on the 100-class dataset using transfer learning, then generates a Kaggle submission using a 3-model ensemble with test-time augmentation.

## 1. Setup & Imports

In [ ]:
import os
import random
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torchvision.transforms import v2 as T
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Check CUDA (Colab/Linux GPU) first, then Apple MPS, then CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

# Change SEED to 50, 51 for the 2nd and 3rd ensemble runs
SEED = 49
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
print(f'Seed: {SEED}')

## 2. Data Loading & Augmentation

Perception Encoder uses 336x336 input resolution with standard ImageNet normalization.

We apply data augmentation during training to help generalize with only ~10 images per class.

In [ ]:
# ---- CHANGE THIS to wherever you unzipped the Kaggle dataset ----
DATA_DIR = './ucsc-cse-144-spring-2026-final-project'   # should contain train/ and test/ folders
# -----------------------------------------------------------------

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# PE uses 336x336 — resize to 384 first to allow random cropping headroom
train_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomCrop(336),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def make_dataset(transform):
    ds = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=transform)
    # ImageFolder sorts folders alphabetically ("10" before "2"), but Kaggle
    # requires label N for folder "N" — remap to numeric order.
    ds.class_to_idx = {cls: int(cls) for cls in ds.classes}
    ds.samples = [(path, int(ds.classes[lbl])) for path, lbl in ds.samples]
    ds.targets  = [int(ds.classes[lbl]) for lbl in ds.targets]
    return ds

# Determine split indices once with a fixed seed so val set is consistent across runs
_base = make_dataset(train_transform)
val_size   = int(0.1 * len(_base))
train_size = len(_base) - val_size
train_idx, val_idx = random_split(
    range(len(_base)), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Create two SEPARATE dataset objects so setting val transform doesn't
# accidentally overwrite the training transform (they share no state)
train_dataset = Subset(make_dataset(train_transform), train_idx.indices)
val_dataset   = Subset(make_dataset(eval_transform),  val_idx.indices)

# num_workers=0 avoids a macOS multiprocessing deadlock with DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)

print(f'Total images: {len(_base)} | Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print(f'Number of classes: {len(_base.classes)}')
print(f'Class mapping (first 5): {dict(list(_base.class_to_idx.items())[:5])}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 3. Build the Model

Perception Encoder is loaded via timm with `num_classes=0` so it returns feature vectors directly.
We attach our own 100-class classification head on top.

Model: `vit_pe_core_large_patch14_336.fb` (ViT-Large, patch 14, 336x336 input)

**Prerequisites:** `pip install timm`

In [ ]:
NUM_CLASSES = 100

class PEClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # num_classes=0 removes timm's built-in head — forward() returns feature vectors
        self.backbone = timm.create_model(
            'vit_pe_core_large_patch14_336.fb',
            pretrained=True,
            num_classes=0
        )
        self.head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(self.backbone.num_features, num_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

model = PEClassifier(NUM_CLASSES)

# Freeze backbone for Phase 1 — only the new head trains
for param in model.backbone.parameters():
    param.requires_grad = False

model = model.to(device)
print(f'Model ready. Feature dimension: {model.backbone.num_features}')
print(f'Head: {model.head}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 4. Phase 1 — Train the Head Only

With the backbone frozen, only the new classification head trains.
We use 10 epochs to give the head enough warmup before unfreezing the full network.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total

criterion = nn.CrossEntropyLoss()

# Phase 1: only train the new head
optimizer_phase1 = optim.Adam(model.head.parameters(), lr=1e-3)

PHASE1_EPOCHS = 10
train_losses, val_losses, train_accs, val_accs = [], [], [], []

print('=== Phase 1: Training head only ===')
for epoch in range(PHASE1_EPOCHS):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer_phase1, criterion)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion)
    train_losses.append(tr_loss); train_accs.append(tr_acc)
    val_losses.append(vl_loss);   val_accs.append(vl_acc)
    print(f'Epoch {epoch+1}/{PHASE1_EPOCHS} | Train Loss: {tr_loss:.3f} Acc: {tr_acc:.3f} | Val Loss: {vl_loss:.3f} Acc: {vl_acc:.3f}')

## 5. Phase 2 — Fine-tune the Whole Network

Unfreeze all layers and fine-tune with a small learning rate and CutMix augmentation.
The cosine scheduler decays LR smoothly over all epochs.

In [ ]:
# Unfreeze ALL layers for full fine-tuning
for param in model.parameters():
    param.requires_grad = True

# Recompute activations during backward instead of storing them — cuts memory significantly
model.backbone.set_grad_checkpointing(True)

print(f'Trainable parameters now: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

cutmix = T.CutMix(num_classes=NUM_CLASSES)

def train_one_epoch_cutmix(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = cutmix(images, labels)
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels.argmax(1)).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

PHASE2_EPOCHS = 25
optimizer_phase2 = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
# T_max matches PHASE2_EPOCHS for one smooth LR decay rather than cycling
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_phase2, T_max=PHASE2_EPOCHS)

best_val_acc = 0.0

print('=== Phase 2: Fine-tuning full network with CutMix ===')
for epoch in range(PHASE2_EPOCHS):
    tr_loss, tr_acc = train_one_epoch_cutmix(model, train_loader, optimizer_phase2, criterion)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion)
    scheduler.step()
    train_losses.append(tr_loss); train_accs.append(tr_acc)
    val_losses.append(vl_loss);   val_accs.append(vl_acc)
    print(f'Epoch {epoch+1}/{PHASE2_EPOCHS} | Train Loss: {tr_loss:.3f} Acc: {tr_acc:.3f} | Val Loss: {vl_loss:.3f} Acc: {vl_acc:.3f}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), f'pe_best_model_{SEED}.pth')
        print(f'  -> New best model saved! Val acc: {vl_acc:.3f}')

print(f'\nBest validation accuracy: {best_val_acc:.3f}')
print(f'Model saved as pe_best_model_{SEED}.pth')

## 6. Plot Training Curves

Include these plots in your report!

In [ ]:
total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS
epochs_range = range(1, total_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, train_losses, label='Train')
ax1.plot(epochs_range, val_losses,   label='Validation')
ax1.axvline(PHASE1_EPOCHS + 0.5, color='gray', linestyle='--', label='Phase 2 start')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss Curve'); ax1.legend()

ax2.plot(epochs_range, train_accs, label='Train')
ax2.plot(epochs_range, val_accs,   label='Validation')
ax2.axvline(PHASE1_EPOCHS + 0.5, color='gray', linestyle='--', label='Phase 2 start')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy Curve'); ax2.legend()

plt.tight_layout()
plt.savefig('pe_training_curves.png', dpi=150)
plt.show()
print('Saved pe_training_curves.png')

## 7. Generate Kaggle Submission

Load all 3 ensemble models, run TTA on each test image, average predictions, write submission.csv.

In [ ]:
import torch.nn.functional as F

# All 3 ensemble model weights — run training 3 times (SEED=49,50,51) to generate these
MODEL_PATHS = ['pe_best_model_49.pth', 'pe_best_model_50.pth', 'pe_best_model_51.pth']
TTA_RUNS = 5  # augmented passes per image per model (plus 1 clean = 6 total)

test_dir = os.path.join(DATA_DIR, 'test')
test_files = sorted(os.listdir(test_dir), key=lambda x: int(x.split('.')[0]))

# TTA uses the same augmentations as training but at 336x336
tta_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomCrop(336),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Accumulate averaged TTA probabilities across all ensemble models
all_probs = [None] * len(test_files)
for model_path in MODEL_PATHS:
    print(f'Running inference with {model_path}...')
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    with torch.no_grad():
        for i, fname in enumerate(test_files):
            img = Image.open(os.path.join(test_dir, fname)).convert('RGB')
            # 1 clean pass + TTA_RUNS augmented passes, averaged
            probs = F.softmax(model(eval_transform(img).unsqueeze(0).to(device)), dim=1)
            for _ in range(TTA_RUNS):
                probs += F.softmax(model(tta_transform(img).unsqueeze(0).to(device)), dim=1)
            probs /= (TTA_RUNS + 1)
            all_probs[i] = probs if all_probs[i] is None else all_probs[i] + probs

predictions = [p.argmax(1).item() for p in all_probs]

# IMPORTANT: Kaggle expects IDs as filenames (e.g. "0.jpg"), not integers
submission = pd.DataFrame({'ID': test_files, 'Label': predictions})
submission = submission.sort_values(by='ID', key=lambda x: x.map(lambda f: int(f.split('.')[0]))).reset_index(drop=True)
submission.to_csv('submission.csv', index=False)

print(f'Saved submission.csv with {len(submission)} predictions')
print(submission.head(10))

## 8. Save Model Weights for Google Drive

After all 3 training runs, upload `pe_best_model_49.pth`, `pe_best_model_50.pth`, and `pe_best_model_51.pth` to Google Drive and paste the link in your GitHub README.

In [ ]:
# Confirm the 3 ensemble model files exist before uploading to Google Drive
for seed in [49, 50, 51]:
    path = f'pe_best_model_{seed}.pth'
    exists = os.path.exists(path)
    print(f'{path}: {"✓ exists" if exists else f"✗ MISSING — run training with SEED={seed}"}')

print('\nFiles to submit/upload:')
print('  submission.csv                        -> Kaggle')
print('  pe_best_model_49/50/51.pth            -> Google Drive (link in README)')
print('  pe_training_curves.png                -> include in report')
print('  this .ipynb file                      -> GitHub repo')